In [ ]:
import pandas as pd
import numpy as np

# 1. Load the data
df = pd.read_csv('Queries.csv')

# 2. Inspect the first few rows
print("Preview of the Data:")
display(df.head())

# 3. Clean the CTR column (convert from 'XX.XX%' string to float)
if df['CTR'].dtype == object:
    df['CTR_num'] = df['CTR'].str.rstrip('%').astype('float') / 100.0
else:
    df['CTR_num'] = df['CTR']

In [1]:
# !pip install google-cloud-bigquery[pandas] pyarrow db-dtypes

In [ ]:
# Extract the first word and convert to lowercase
['first_word'] = df['Top queries'].str.split().str[0].str.lower()

# Group by the first word and aggregate the metrics
first_word_summary = df.groupby('first_word').agg(
    total_impressions=('Impressions', 'sum'),
    total_clicks=('Clicks', 'sum'),
    query_count=('Top queries', 'count'),
    avg_position=('Position', 'mean')
).reset_index()

# Calculate overall totals
total_all_impressions = first_word_summary['total_impressions'].sum()
total_all_clicks = first_word_summary['total_clicks'].sum()

# Add the percentage contribution columns
first_word_summary['%_of_total_impressions'] = (first_word_summary['total_impressions'] / total_all_impressions) * 100
first_word_summary['%_of_total_clicks'] = (first_word_summary['total_clicks'] / total_all_clicks) * 100

# Format the percentages for better readability (optional)
first_word_summary['%_of_total_impressions'] = first_word_summary['%_of_total_impressions'].round(2).astype(str) + '%'
first_word_summary['%_of_total_clicks'] = first_word_summary['%_of_total_clicks'].round(2).astype(str) + '%'

# Sort by highest impression volume
first_word_summary = first_word_summary.sort_values(by='total_impressions', ascending=False)

print("Top Action Words by Volume and Contribution:")
display(first_word_summary.head(10))

In [ ]:
df.to_clipboard()

In [ ]:
df.head()

In [ ]:
df[df['first_word']=='can'].sort_values(by=['Impressions'],ascending=False).head(10)

In [ ]:
# Define the categorization logic
def categorize_query(query):
    query_lower = str(query).lower()
    if query_lower in ['yes', 'yes please', 'no', 'ok', 'sure', 'yeah', 'yea', 'thanks', 'ty']:
        return 'Conversational Responses'
    elif query_lower.startswith('compare'):
        return 'Comparisons'
    elif query_lower.startswith('list'):
        return 'Listings / Catalogs'
    elif any(query_lower.startswith(w) for w in ['can you', 'explain', 'how', 'is that', 'estimate']):
        return 'Educational / Feasibility'
    elif any(query_lower.startswith(w) for w in ['find', 'give', 'show', 'tell']):
        return 'Direct Action / Navigational'
    else:
        return 'Other'

# Apply the function to create a new 'theme' column
df['theme'] = df['Top queries'].apply(categorize_query)

# Aggregate metrics by these new themes
theme_summary = df.groupby('theme').agg(
    total_impressions=('Impressions', 'sum'),
    total_clicks=('Clicks', 'sum'),
    query_count=('Top queries', 'count'),
    avg_position=('Position', 'mean')
).reset_index()

# Calculate overall totals for themes
theme_total_impressions = theme_summary['total_impressions'].sum()
theme_total_clicks = theme_summary['total_clicks'].sum()

# Add the percentage contribution columns
theme_summary['%_of_total_impressions'] = (theme_summary['total_impressions'] / theme_total_impressions) * 100
theme_summary['%_of_total_clicks'] = (theme_summary['total_clicks'] / theme_total_clicks) * 100

# Format the percentages for better readability (optional)
theme_summary['%_of_total_impressions'] = theme_summary['%_of_total_impressions'].round(2).astype(str) + '%'
theme_summary['%_of_total_clicks'] = theme_summary['%_of_total_clicks'].round(2).astype(str) + '%'

theme_summary = theme_summary.sort_values(by='total_impressions', ascending=False)

print("Performance and Contribution by AI Search Theme:")
display(theme_summary)

In [ ]:
# Isolate only the 'compare' queries
compare_df = df[df['theme'] == 'Comparisons'].copy()

# Sort by Clicks to see what is driving the most traffic
compare_df = compare_df.sort_values(by='Clicks', ascending=False)

# Calculate the total clicks and impressions strictly for the 'Compare' category
compare_total_clicks = compare_df['Clicks'].sum()
compare_total_impressions = compare_df['Impressions'].sum()

# Add columns to show the percentage contribution of each query within this category
compare_df['%_of_compare_clicks'] = (compare_df['Clicks'] / compare_total_clicks) * 100
compare_df['%_of_compare_impressions'] = (compare_df['Impressions'] / compare_total_impressions) * 100

# Format the percentages for better readability
compare_df['%_of_compare_clicks'] = compare_df['%_of_compare_clicks'].round(2).astype(str) + '%'
compare_df['%_of_compare_impressions'] = compare_df['%_of_compare_impressions'].round(2).astype(str) + '%'

print("Top 10 Comparison Queries by Contribution:")
display(compare_df[['Top queries', 'Clicks', 'Impressions', '%_of_compare_clicks', '%_of_compare_impressions', 'Position']].head(10))

# Find mentions of specific high-value models in the comparison data
prado_comparisons = compare_df[compare_df['Top queries'].str.contains('prado', case=False, na=False)]
print(f"\nTotal Prado Comparison Queries: {len(prado_comparisons)}")
print(f"Total Prado Comparison Clicks: {prado_comparisons['Clicks'].sum()}")

In [ ]:
# prado_comparisons

In [ ]:
compare_df['CTR']=""
compare_df['CTR']=compare_df['Clicks']/compare_df['Impressions']

In [ ]:
compare_df.head(20)